In [3]:
from sklearn.datasets import load_iris, fetch_california_housing
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import numpy as np

In [4]:
# Dataset 1: d
file_1 = "https://raw.githubusercontent.com/martin-paz-y/machine-learning-project/refs/heads/main/job_train.csv"
df = pd.read_csv(file_1)
df.head()

,title,location,description,requirements,telecommuting,has_company_logo,has_questions,fraudulent
0,Architect (Middleware - MQ) - Kuwait,"KW, KU,","On behalf of our client, a well known multinat...",-Working technical knowledge of IT systems and...,0,1,0,0
1,Interviewing Now for Sales Rep Positions -- wi...,"US, TX, Corpus Christi","We are Argenta Field Solutions, a rapidly expa...",#NAME?,0,1,0,0
2,Process Controls Staff Engineer - Foxboro I/A ...,"US, TX, USA Southwest",Experienced Process Controls Staff Engineer is...,At least 10 years of degreed professional expe...,0,0,0,0
3,Experienced Telemarketer Wanted - Digital Solu...,"AU, NSW,",If you have a passion for people and love to s...,"Responsibilities - Prospecting, following up a...",0,1,0,0
4,Senior Network Engineer,"GB, ENG, London",As the successful Senior Network Engineer you ...,Essential skills:•Juniper switching/routing/se...,0,1,0,0


In [5]:
df.isnull().sum()

title                  0
location             157
description            1
requirements        1326
telecommuting          0
has_company_logo       0
has_questions          0
fraudulent             0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(105)

In [7]:
import re
import pandas as pd
import numpy as np

def clean_text(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).lower()
    texto = re.sub(r"<.*?>", " ", texto)         # remove HTML tags
    texto = re.sub(r"[^a-z\s]", " ", texto)      # keep letters + spaces only
    texto = re.sub(r"\s+", " ", texto)           # normalize spaces
    return texto.strip()

columnas_texto = ["title", "description", "requirements"]

for col in columnas_texto:
    df[col] = df[col].apply(clean_text)

df.head()


,title,location,description,requirements,telecommuting,has_company_logo,has_questions,fraudulent
0,architect middleware mq kuwait,"KW, KU,",on behalf of our client a well known multinati...,working technical knowledge of it systems and ...,0,1,0,0
1,interviewing now for sales rep positions with ...,"US, TX, Corpus Christi",we are argenta field solutions a rapidly expan...,name,0,1,0,0
2,process controls staff engineer foxboro i a tr...,"US, TX, USA Southwest",experienced process controls staff engineer is...,at least years of degreed professional experie...,0,0,0,0
3,experienced telemarketer wanted digital solutions,"AU, NSW,",if you have a passion for people and love to s...,responsibilities prospecting following up and ...,0,1,0,0
4,senior network engineer,"GB, ENG, London",as the successful senior network engineer you ...,essential skills juniper switching routing sec...,0,1,0,0


In [8]:
df["location"].value_counts().head(20)



location
GB, LND, London          367
US, NY, New York         331
GR, I, Athens            244
US, CA, San Francisco    241
US, ,                    180
US, TX, Houston          147
US, DC, Washington       129
US, IL, Chicago          127
NZ, N, Auckland          110
DE, BE, Berlin           103
US, CA, Los Angeles       91
US, TX, Austin            88
GB, , London              83
US, CA, San Diego         71
US, OR, Portland          71
GB, ,                     70
US, GA, Atlanta           69
CA, ON, Toronto           62
US, PA, Philadelphia      60
GB, LND,                  55
Name: count, dtype: int64

In [9]:
#normalzie text

df["location"] = df["location"].str.lower().str.strip()



In [10]:
df[["telecommuting", "has_company_logo", "has_questions"]].value_counts().head(10)

telecommuting  has_company_logo  has_questions
0              1                 1                3718
                                 0                3098
               0                 0                1286
                                 1                 447
1              1                 1                 163
                                 0                 123
               0                 0                  77
                                 1                  28
Name: count, dtype: int64

In [11]:
df.head()

,title,location,description,requirements,telecommuting,has_company_logo,has_questions,fraudulent
0,architect middleware mq kuwait,"kw, ku,",on behalf of our client a well known multinati...,working technical knowledge of it systems and ...,0,1,0,0
1,interviewing now for sales rep positions with ...,"us, tx, corpus christi",we are argenta field solutions a rapidly expan...,name,0,1,0,0
2,process controls staff engineer foxboro i a tr...,"us, tx, usa southwest",experienced process controls staff engineer is...,at least years of degreed professional experie...,0,0,0,0
3,experienced telemarketer wanted digital solutions,"au, nsw,",if you have a passion for people and love to s...,responsibilities prospecting following up and ...,0,1,0,0
4,senior network engineer,"gb, eng, london",as the successful senior network engineer you ...,essential skills juniper switching routing sec...,0,1,0,0


In [12]:
df["fraudulent"].value_counts()
df[["telecommuting","has_company_logo","has_questions"]].nunique()


telecommuting       2
has_company_logo    2
has_questions       2
dtype: int64

In [13]:
#2) Crear una columna de texto unificada (recomendado) Así le das al modelo más señal y simplificas TF-IDF.

df["text"] = (df["title"] + " " + df["description"] + " " + df["requirements"]).str.strip()


# Logistic Regression + TF-IDF

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

X = df[["text", "telecommuting", "has_company_logo", "has_questions"]]
y = df["fraudulent"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(max_features=20000, ngram_range=(1,2)), "text"),
        # Escalado solo para las columnas numéricas (aquí son 0/1, no hace daño)
        ("bin", StandardScaler(with_mean=False), ["telecommuting","has_company_logo","has_questions"])
    ],
    remainder="drop"
)

model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

model.fit(X_train, y_train)
preds = model.predict(X_test)

print(confusion_matrix(y_test, preds))
print(classification_report(y_test, preds))


[[1616   81]
 [  18   73]]
              precision    recall  f1-score   support

           0       0.99      0.95      0.97      1697
           1       0.47      0.80      0.60        91

    accuracy                           0.94      1788
   macro avg       0.73      0.88      0.78      1788
weighted avg       0.96      0.94      0.95      1788



### Same model but now we try a different threshold more aligned with fraud.

In [15]:
X = df[["text", "telecommuting", "has_company_logo", "has_questions"]]
y = df["fraudulent"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(max_features=20000, ngram_range=(1,2)), "text"),
        # Escalado solo para las columnas numéricas (aquí son 0/1, no hace daño)
        ("bin", StandardScaler(with_mean=False), ["telecommuting","has_company_logo","has_questions"])
    ],
    remainder="drop"
)

model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

model.fit(X_train, y_train)
# 1️⃣ Obtener probabilidades de fraude
probs = model.predict_proba(X_test)[:, 1]

# 2️⃣ DEFINIR EL UMBRAL (AQUÍ ES DONDE LO CAMBIAS)
threshold = 0.3   # prueba 0.3, 0.4, 0.5, 0.7

# 3️⃣ Convertir probabilidades a 0 / 1 usando el umbral
preds = (probs >= threshold).astype(int)

print(confusion_matrix(y_test, preds))
print(classification_report(y_test, preds))

[[1500  197]
 [   8   83]]
              precision    recall  f1-score   support

           0       0.99      0.88      0.94      1697
           1       0.30      0.91      0.45        91

    accuracy                           0.89      1788
   macro avg       0.65      0.90      0.69      1788
weighted avg       0.96      0.89      0.91      1788



# Decision tree model

🌳 Decision Tree para detección de fraude
🔹 Por qué Decision Tree aquí

Es un modelo supervisado de clasificación

Fácil de interpretar

Sirve como baseline no lineal

Suele sobreajustar → por eso controlamos la profundidad

In [16]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# Features y target
X = df[["text", "telecommuting", "has_company_logo", "has_questions"]]
y = df["fraudulent"].astype(int)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Preprocesamiento (igual que antes)
preprocess = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(max_features=20000, ngram_range=(1,2)), "text"),
        ("bin", StandardScaler(with_mean=False),
         ["telecommuting", "has_company_logo", "has_questions"])
    ]
)

# Decision Tree (CONTROLANDO overfitting)
dt_model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", DecisionTreeClassifier(
        max_depth=15,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42
    ))
])

# Entrenamiento
dt_model.fit(X_train, y_train)

# Predicciones
preds_dt = dt_model.predict(X_test)

# Evaluación
print(confusion_matrix(y_test, preds_dt))
print(classification_report(y_test, preds_dt))


[[1492  205]
 [  21   70]]
              precision    recall  f1-score   support

           0       0.99      0.88      0.93      1697
           1       0.25      0.77      0.38        91

    accuracy                           0.87      1788
   macro avg       0.62      0.82      0.66      1788
weighted avg       0.95      0.87      0.90      1788



## Comparación de Modelos, Logistic Regression vs Decision Tree

| Modelo            | Recall fraude | Precision fraude | Accuracy |
| ----------------- | ------------- | ---------------- | -------- |
| Logistic (0.5)    | ~0.80         | ~0.47            | ~0.94    |
| Logistic (0.3)    | ~0.91         | ~0.30            | ~0.89    |
| **Decision Tree** | **0.77**      | **0.25**         | **0.87** |


🆚 Comparación directa con tu Logistic Regression
Modelo	Recall fraude	Precision fraude	Accuracy
Logistic (0.5)	~0.80	~0.47	~0.94
Logistic (0.3)	~0.91	~0.30	~0.89
Decision Tree	0.77	0.25	0.87

👉 El Decision Tree es inferior en todos los aspectos clave.

✍️ Texto listo para entregar

The Decision Tree model shows a relatively good recall for fraud detection, but it suffers from very low precision and a high number of false positives.
Compared to Logistic Regression with TF-IDF, the tree generalizes worse and is less suitable for high-dimensional text data.
Therefore, Logistic Regression remains the preferred model for this task.

🔥 Mensaje importante (esto suma nota)

Que un Decision Tree funcione peor no es un fallo:
👉 es la conclusión correcta para NLP con TF-IDF.

Si quieres, el siguiente paso natural sería:

Random Forest para texto

XGBoost / LightGBM

o calibración de probabilidades

### Random Forest (Baseline)

A Random Forest model is trained using the same preprocessing pipeline from previous models.
This is important because the dataset contains text. Text cannot be used directly by scikit-learn models, so it must first be transformed into numerical features using TF-IDF.

To avoid data leakage and ensure the same transformations are applied consistently, I reuse the existing `preprocess` object inside a Pipeline:
- Step 1: `prep` converts the text column into TF-IDF features and keeps structured binary features.
- Step 2: `clf` trains a Random Forest classifier on the resulting numeric matrix.

After training, the model is evaluated on the test set using Accuracy, Confusion Matrix, and Precision/Recall/F1-score.


In [22]:
###
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

rf_model = Pipeline(steps=[
    ("prep", preprocess),  # <-- el mismo ColumnTransformer de tu Logistic/Tree
    ("clf", RandomForestClassifier(
        random_state=42,
        n_estimators=300,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))
###


Random Forest Accuracy: 0.9731543624161074
[[1697    0]
 [  48   43]]
              precision    recall  f1-score   support

           0       0.97      1.00      0.99      1697
           1       1.00      0.47      0.64        91

    accuracy                           0.97      1788
   macro avg       0.99      0.74      0.81      1788
weighted avg       0.97      0.97      0.97      1788



### Hyperparameter Tuning with GridSearchCV + Cross Validation

The baseline Random Forest uses default settings, but its performance can be improved by tuning key hyperparameters.
In Random Forest, important parameters include:
- `n_estimators`: number of trees (more trees can improve stability)
- `max_depth`: maximum depth of each tree (controls complexity and overfitting)
- `min_samples_split` / `min_samples_leaf`: minimum samples needed to create or keep nodes (regularization)

To search for the best combination, I use **GridSearchCV** with **StratifiedKFold cross-validation**.
Cross-validation splits the training data multiple times and evaluates each configuration across different folds, which helps ensure the selected model generalizes well.

Since this is a fraud detection problem (imbalanced classes), I optimize the **F1-score** as a balance between Precision and Recall.


In [18]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

param_grid = {
    "clf__n_estimators": [200, 400],
    "clf__max_depth": [None, 20, 40],
    "clf__min_samples_split": [2, 5],
    "clf__min_samples_leaf": [1, 5]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",   # si quieres priorizar fraude -> mejor "recall"
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

best_rf_model = grid_search.best_estimator_

print("Best parameters found:", grid_search.best_params_)


Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best parameters found: {'clf__max_depth': 40, 'clf__min_samples_leaf': 5, 'clf__min_samples_split': 2, 'clf__n_estimators': 400}


### Evaluation of the Tuned Random Forest

After GridSearchCV finishes, the best model configuration is stored in `best_rf_model`.
This tuned model is then evaluated on the unseen test set.

The evaluation includes:
- Accuracy: overall correctness (not always the best metric for imbalance)
- Confusion Matrix: detailed view of False Positives and False Negatives
- Classification Report: Precision, Recall, and F1-score for each class

This helps confirm whether tuning improves fraud detection performance compared to the baseline model.


In [19]:
y_pred_best_rf = best_rf_model.predict(X_test)

print("Tuned Random Forest Accuracy:", accuracy_score(y_test, y_pred_best_rf))
print(confusion_matrix(y_test, y_pred_best_rf))
print(classification_report(y_test, y_pred_best_rf))


Tuned Random Forest Accuracy: 0.9731543624161074
[[1691    6]
 [  42   49]]
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1697
           1       0.89      0.54      0.67        91

    accuracy                           0.97      1788
   macro avg       0.93      0.77      0.83      1788
weighted avg       0.97      0.97      0.97      1788

